# JWST_DINO — interactive training / smoke tests

Self-contained Lightning re-implementation of DINOv2 pre-training (no `dinov2` import).
Run cells below for quick interactive trials; use `sbatch/train_dev.sh` and
`sbatch/train_distribute.sh` for the real multi-GPU / multi-node runs.

Full run (matches the old `ps6_st3_distribute` config):
```bash
python trainer.py fit --config jwst_dino.yaml --trainer.devices=4
```

## 1. Smoke test — `fast_dev_run` (5 train + 5 val steps, 1 GPU)
Checks the whole pipeline wires up and losses are finite; no checkpoints written.

In [6]:
!python trainer.py fit --config jwst_dino.yaml \
    --trainer.devices=1 --trainer.num_nodes=1 \
    --trainer.fast_dev_run=5 --data.num_workers=2 --data.batch_size=8

Seed set to 0
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
Running in `fast_dev_run` mode: will run the requested loop using 5 batch(es). Logging and checkpointing is suppressed.
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

###################################
Using data augmentation parameters:
local_crops_number: 8
global_crops_size: 72
local_crops_size: 36
center_crop_size: 100
noise_w: 1.5  noise_s_max: None
###################################
Model input image: 72x72
####################################

JWST [train] — 683507 / 759453 cutouts (f150w)
JWST [val] — 37973 / 759453 cutouts (f150w)
/home/yachen

## 2. Short interactive run — a few tiny epochs on 1 GPU
Exercises validation + teacher-backbone export + checkpointing on a small schedule.

In [8]:
!python trainer.py fit --config jwst_dino.yaml \
    --trainer.devices=1 --trainer.num_nodes=1 \
    --trainer.max_epochs=50 --trainer.limit_train_batches=100 \
    --trainer.check_val_every_n_epoch=2 --data.num_workers=8 --data.batch_size=64

Seed set to 0
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

###################################
Using data augmentation parameters:
local_crops_number: 8
global_crops_size: 72
local_crops_size: 36
center_crop_size: 100
noise_w: 1.5  noise_s_max: None
###################################
Model input image: 72x72
####################################

JWST [train] — 683507 / 759453 cutouts (f150w)
JWST [val] — 37973 / 759453 cutouts (f150w)
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/yacheng/nexus/ssl_outthere/.pixi/envs/default/lib/python3.11/site-packages/lightning

## 3. Inspect an exported teacher backbone
The `.pth` uses the dinov2 `{"teacher": state_dict}` layout the benchmark loaders expect.

In [9]:
import glob, torch
ckpts = sorted(glob.glob('outputs/jwst_dino_ps6_st3/version_*/eval/*/teacher_checkpoint.pth'))
print(f'{len(ckpts)} teacher checkpoints found')
if ckpts:
    sd = torch.load(ckpts[-1], map_location='cpu')['teacher']
    print(ckpts[-1])
    print('params:', sum(v.numel() for v in sd.values()))
    print('first keys:', list(sd.keys())[:5])

4 teacher checkpoints found


/tmp/ipykernel_4060698/1585443641.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(ckpts[-1], map_location='cpu')['teacher']


outputs/jwst_dino_ps6_st3/version_0/eval/800/teacher_checkpoint.pth
params: 38136832
first keys: ['cls_token', 'mask_token', 'register_tokens', 'pos_embed', 'patch_embed.proj.weight']
